# Semantic Search in ChromaDB

In [23]:
# !pip install chromadb

In [24]:
import chromadb
import pandas as pd

In [25]:
df = pd.read_csv("medium_post_titles.csv")

# Remove missing values from dataframe
df = df.dropna() 

# First 1000 rows
df = df.head(1000)

df = df[~df["subtitle_truncated_flag"]]

topics_of_interest = ['artificial-intelligence', 'data-science', 'machine-learning']

df = df[df['category'].isin(topics_of_interest)]

df['text'] = df['title']  + df['subtitle']

df['meta'] = df.apply( lambda x: {
    'text': x['text'],
    'category': x['category']  
}, axis=1)

df.head(5)

,category,title,subtitle,subtitle_truncated_flag,text,meta
4,artificial-intelligence,"""Can I Train my Model on Your Computer?""",How we waste computational resources and how t...,False,"""Can I Train my Model on Your Computer?""How we...","{'text': '""Can I Train my Model on Your Comput..."
289,data-science,(Robot) data scientists as a service,Automating data science with symbolic regressi...,False,(Robot) data scientists as a serviceAutomating...,{'text': '(Robot) data scientists as a service...
448,data-science,10 Free tools to get started with Data Visuali...,Jump right into the Data Visualisation process...,False,10 Free tools to get started with Data Visuali...,{'text': '10 Free tools to get started with Da...
454,data-science,10 Great Programming Projects to Improve Your ...,"Improve your skills in web development, progra...",False,10 Great Programming Projects to Improve Your ...,{'text': '10 Great Programming Projects to Imp...
487,machine-learning,10 Lessons Learned From Participating in Googl...,"Quick, Draw! Doodle Recognition Challenge was ...",False,10 Lessons Learned From Participating in Googl...,{'text': '10 Lessons Learned From Participatin...


## In-Memory Chroma DB Setup

In [26]:
chroma_client = chromadb.Client()

In [27]:
# Inserting data
articles_collection = chroma_client.get_or_create_collection(name="medium-articles")

In [28]:
articles_collection.upsert(
    ids=[f"{x}" for x in df.index.tolist()], # Use list comprehension to convert index from int to string
    documents=df["text"].tolist(),
    metadatas=df["meta"].tolist()
)

In [31]:
# Querying data
query = "Best data science library?"

articles_collection.query(query_texts=query, n_results=2)

{'ids': [['448', '587']],
 'embeddings': None,
 'documents': [['10 Free tools to get started with Data Visualisation-Easily & Instantly.Jump right into the Data Visualisation process with these easy and intuitive tools.',
   '10 Steps to Teaching Data Science WellA resource for data science instructors.']],
 'uris': None,
 'data': None,
 'metadatas': [[{'category': 'data-science',
    'text': '10 Free tools to get started with Data Visualisation-Easily & Instantly.Jump right into the Data Visualisation process with these easy and intuitive tools.'},
   {'category': 'data-science',
    'text': '10 Steps to Teaching Data Science WellA resource for data science instructors.'}]],
 'distances': [[0.9389312863349915, 0.9592275023460388]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [32]:
query = "How to learn artificial intelligence?"

articles_collection.query(query_texts=query, n_results=2)

{'ids': [['752', '505']],
 'embeddings': None,
 'documents': [['10 things about AI every newsroom should knowPreparing your newsroom for the artificial intelligence revolution',
   '10 New Things I Learnt from fast.ai v3Learning points from 3 weeks of taking the course']],
 'uris': None,
 'data': None,
 'metadatas': [[{'category': 'artificial-intelligence',
    'text': '10 things about AI every newsroom should knowPreparing your newsroom for the artificial intelligence revolution'},
   {'category': 'artificial-intelligence',
    'text': '10 New Things I Learnt from fast.ai v3Learning points from 3 weeks of taking the course'}]],
 'distances': [[1.0295445919036865, 1.0543593168258667]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}